In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, avg, date_sub, current_date, rand, when, broadcast

In [2]:
# Create a new SparkSession
spark = (SparkSession
         .builder
         .appName("optimize-data-shuffles")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "2g")
         .getOrCreate())

# Set log level to ERROR
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/01 14:14:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Create some sample data frames
# A large data frame with 1 million rows and two columns: id and value
large_df = (spark.range(0, 1000000)
            .withColumn("date", date_sub(current_date(), (rand() * 365).cast("int")))
            .withColumn("age", (rand() * 100).cast("int"))
            .withColumn("salary", 100*(rand() * 100).cast("int"))
            .withColumn("gender", when((rand() * 2).cast("int") == 0, "M").otherwise("F"))
            .withColumn("grade", 
                        when((rand() * 5).cast("int") == 0, "IC")
                        .when((rand() * 5).cast("int") == 1, "IC-2")
                        .when((rand() * 5).cast("int") == 2, "M1")
                        .when((rand() * 5).cast("int") == 3, "M2")
                        .when((rand() * 5).cast("int") == 4, "IC-3")
                        .otherwise("M3")))
large_df.show(5)

+---+----------+---+------+------+-----+
| id|      date|age|salary|gender|grade|
+---+----------+---+------+------+-----+
|  0|2024-10-08| 66|  3600|     F|   M3|
|  1|2024-07-23| 16|  9100|     F| IC-3|
|  2|2024-03-27| 16|  3900|     F|   M3|
|  3|2024-05-29| 89|  1800|     F|   M2|
|  4|2024-03-20| 92|  5500|     F|   M3|
+---+----------+---+------+------+-----+
only showing top 5 rows



In [4]:
df_filtered = large_df.filter(col("age") >= 55)

In [5]:
df_mapped = df_filtered.withColumn("bonus", col("salary") * 1.1)

In [6]:
df_aggregated = df_mapped.groupBy("age").agg(avg("bonus"))

In [7]:
df_aggregated.show()

+---+------------------+
|age|        avg(bonus)|
+---+------------------+
| 85| 5404.112789526686|
| 65| 5422.651167533394|
| 78|5455.1469118528885|
| 81| 5431.577283950617|
| 76|5405.8498611662035|
| 91| 5399.974457215836|
| 93| 5459.303828940825|
| 86| 5437.285914365746|
| 94| 5450.965695022804|
| 57| 5468.215518992484|
| 96| 5478.041369754553|
| 92| 5403.414365746298|
| 64| 5481.357227958697|
| 61| 5408.034420289855|
| 88| 5413.320394147507|
| 72|5462.6379690949225|
| 59| 5422.449779027722|
| 55| 5412.180797755062|
| 84| 5440.977488744372|
| 87| 5511.387763371151|
+---+------------------+
only showing top 20 rows



In [8]:
df_aggregated.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[age#5], functions=[avg(bonus#52)])
   +- Exchange hashpartitioning(age#5, 200), ENSURE_REQUIREMENTS, [plan_id=123]
      +- HashAggregate(keys=[age#5], functions=[partial_avg(bonus#52)])
         +- Project [age#5, (cast(salary#9 as double) * 1.1) AS bonus#52]
            +- Filter (isnotnull(age#5) AND (age#5 >= 55))
               +- Project [age#5, (cast((rand(-6086861404426379402) * 100.0) as int) * 100) AS salary#9]
                  +- Project [cast((rand(-8329055057124076450) * 100.0) as int) AS age#5]
                     +- Range (0, 1000000, step=1, splits=2)




In [9]:
df2 = spark.createDataFrame([
    (25, "A"),
    (30, "B"),
    (35, "C"),
    (40, "D"),
    (45, "E"),
    (50, "F"),
    (55, "G"),
    (60, "H"),
    (65, "I"),
    (70, "J")],
    ["age", "level"]
)

In [10]:
df_joined = large_df.join(broadcast(df2), "age")

In [11]:
df_aggregated = df_joined.groupBy("level").avg("salary")

In [12]:
df_aggregated.show()

+-----+-----------------+
|level|      avg(salary)|
+-----+-----------------+
|    F|4947.866732968672|
|    E|4963.956322511455|
|    B|4950.140224358975|
|    D|4958.232409168251|
|    C|5010.136847440446|
|    J|4946.273447687832|
|    A|4959.176482432296|
|    G| 4920.16436159551|
|    I|4929.682879575813|
|    H|4990.048348106366|
+-----+-----------------+



In [13]:
df_aggregated.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[level#87], functions=[avg(salary#9)])
   +- Exchange hashpartitioning(level#87, 200), ENSURE_REQUIREMENTS, [plan_id=311]
      +- HashAggregate(keys=[level#87], functions=[partial_avg(salary#9)])
         +- Project [salary#9, level#87]
            +- BroadcastHashJoin [cast(age#5 as bigint)], [age#86L], Inner, BuildRight, false
               :- Filter isnotnull(age#5)
               :  +- Project [age#5, (cast((rand(-6086861404426379402) * 100.0) as int) * 100) AS salary#9]
               :     +- Project [cast((rand(-8329055057124076450) * 100.0) as int) AS age#5]
               :        +- Range (0, 1000000, step=1, splits=2)
               +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=306]
                  +- Filter isnotnull(age#86L)
                     +- Scan ExistingRDD[age#86L,level#87]




In [14]:
df_repartitioned = large_df.repartition(col("gender"))

In [15]:
df_repartitioned_by_range = large_df.repartitionByRange(5, col("age"))

In [16]:
large_df.explain()

== Physical Plan ==
*(1) Project [id#0L, date#2, age#5, salary#9, gender#14, CASE WHEN (cast((rand(3451796573625257076) * 5.0) as int) = 0) THEN IC WHEN (cast((rand(-2619993627747595143) * 5.0) as int) = 1) THEN IC-2 WHEN (cast((rand(5011442441506356043) * 5.0) as int) = 2) THEN M1 WHEN (cast((rand(7432489719469921642) * 5.0) as int) = 3) THEN M2 WHEN (cast((rand(7679098802775812609) * 5.0) as int) = 4) THEN IC-3 ELSE M3 END AS grade#20]
+- *(1) Project [id#0L, date#2, age#5, salary#9, CASE WHEN (cast((rand(-1595611425252378001) * 2.0) as int) = 0) THEN M ELSE F END AS gender#14]
   +- *(1) Project [id#0L, date#2, age#5, (cast((rand(-6086861404426379402) * 100.0) as int) * 100) AS salary#9]
      +- *(1) Project [id#0L, date#2, cast((rand(-8329055057124076450) * 100.0) as int) AS age#5]
         +- *(1) Project [id#0L, date_sub(2025-03-01, cast((rand(-5765831752670284425) * 365.0) as int)) AS date#2]
            +- *(1) Range (0, 1000000, step=1, splits=2)




In [17]:
df_repartitioned.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange hashpartitioning(gender#14, 200), REPARTITION_BY_COL, [plan_id=363]
   +- Project [id#0L, date#2, age#5, salary#9, gender#14, CASE WHEN (cast((rand(3451796573625257076) * 5.0) as int) = 0) THEN IC WHEN (cast((rand(-2619993627747595143) * 5.0) as int) = 1) THEN IC-2 WHEN (cast((rand(5011442441506356043) * 5.0) as int) = 2) THEN M1 WHEN (cast((rand(7432489719469921642) * 5.0) as int) = 3) THEN M2 WHEN (cast((rand(7679098802775812609) * 5.0) as int) = 4) THEN IC-3 ELSE M3 END AS grade#20]
      +- Project [id#0L, date#2, age#5, salary#9, CASE WHEN (cast((rand(-1595611425252378001) * 2.0) as int) = 0) THEN M ELSE F END AS gender#14]
         +- Project [id#0L, date#2, age#5, (cast((rand(-6086861404426379402) * 100.0) as int) * 100) AS salary#9]
            +- Project [id#0L, date#2, cast((rand(-8329055057124076450) * 100.0) as int) AS age#5]
               +- Project [id#0L, date_sub(2025-03-01, cast((rand(-576583175267028